In [ ]:
# Copyright 2025 DeepMind Technologies Limited. All Rights Reserved.

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/google/genai-processors/blob/main/notebooks/research_example.ipynb)
[![License](https://img.shields.io/badge/License-Apache_2.0-blue.svg)](https://github.com/google-gemini/genai-processors/blob/main/LICENSE)

# 📖 Research Agent Example

This notebook demonstrates how to build a research agent using the modular
components of the `genai-processors` library.

You will see how a complex task, like researching a topic, can be broken down
into a series of smaller, reusable processors. We will explore each component
individually and then combine them into a powerful, automated research pipeline.

In this notebook, we will cover:

*   **Setup**: Getting your environment ready and configuring an API key.
*   **Individual Processors**: Exploring the building blocks `TopicGenerator`,
    `TopicResearcher`, and `TopicVerbalizer`.
*   **Chaining**: Learning how to connect processors to create a seamless
    workflow.
*   **The Complete Agent**: Running the `ResearchAgent` to see how all the
    pieces come together for an end-to-end task.

Note: You will need to authorize colab to use your GitHub account, in order to
import the research example.

## 🍳 Setup

In [ ]:
# {display-mode: "form"}
# @markdown Run this cell to import libraries and perform initializations.

!pip install genai_processors

from genai_processors import content_api
from genai_processors import processor
from genai_processors import streams
from genai_processors.core import jinja_template
from genai_processors.examples import research
from google.colab import userdata
from IPython.display import Markdown, display

ProcessorPart = processor.ProcessorPart


def render_part(part: ProcessorPart) -> None:
  if part.substream_name == "status":
    display(Markdown(f"--- \n *Status*: {part.text}"))
  else:
    try:
      display(Markdown(part.text))
    except Exception:
      display(Markdown(f" {part.text} "))

## 🔐 Auth

To use the research processors, you will need an API key. If you have not done
so already, obtain your API key from Google AI Studio, and import it as a secret
in Colab (recommended) or directly set it below.

In [ ]:
GOOGLE_API_KEY = None

try:
  GOOGLE_API_KEY = userdata.get('GOOGLE_API_KEY')
except Exception:
  print('Failed to obtain `GOOGLE_API_KEY`.')

## 🏗 Processors

In [ ]:
USER_PROMPT = "Research the best things about owning dalmatians!"  # @param { "type": "string" }

### ✍ `TopicGenerator`

The `TopicGenerator` processor generates a list of research topics, given the
user's content.

In [ ]:
p_generator = research.TopicGenerator(api_key=GOOGLE_API_KEY)

topic_parts = []
async for content_part in p_generator(USER_PROMPT):
  if content_part.mimetype == 'application/json; type=Topic':
    topic_parts.append(content_part)
  else:
    render_part(content_part)

### 🔍 `TopicResearcher`

Next, we add `TopicResearcher` to `TopicGenerator` to generate `Topic` objects.

In [ ]:
topics = []
p_researcher = research.TopicResearcher(api_key=GOOGLE_API_KEY)

pipeline = p_generator + p_researcher

async for content_part in pipeline(USER_PROMPT):
  if content_part.mimetype == 'application/json; type=Topic':
    topics.append(content_part.get_dataclass(research.interfaces.Topic))
  elif content_part.substream_name == 'status':
    render_part(content_part)

print(f'Pipeline produced {len(topics)} `Topic` `ProcessorParts`:\n\n')

for t in topics:
  print(t)

### 🗣 `TopicVerbalizer`

A Jinja2 processor is used to convert `TopicResearch` parts into human-readable
research text.

In [ ]:
p_verbalizer = jinja_template.RenderDataClass(
    template_str=(
        "## {{ data.topic }}\n"
        "*{{ data.relationship_to_user_content }}*"
        "{% if data.research_text|trim != '' %}"
        "\n\n### Research\n\n{{ data.research_text }}"
        "{% endif %}"
    ),
    data_class=research.interfaces.Topic,
)

pipeline = p_generator + p_researcher + p_verbalizer

async for content_part in pipeline(USER_PROMPT):
  # We exclude printing status to demonstrate the verbalization.
  if content_part.substream_name != "status":
    render_part(content_part)

## 🤖 Agent

Now we have all our building blocks, we can chain these together inside our
agent, resulting in a seamless flow of Content.

In [ ]:
output_parts = content_api.ProcessorContent()
async for content_part in research.ResearchAgent(api_key=GOOGLE_API_KEY)(
    USER_PROMPT
):
  if content_part.substream_name == 'status':
    render_part(content_part)
  output_parts += content_part

render_part(ProcessorPart(f"""# Final synthesized research

{content_api.as_text(output_parts, substream_name='')}"""))